In [1]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from tqdm import tqdm
from archetypes_re2020.data.remove_extreme_outliers import outliers

In [2]:
with open("/home/thibault.chevilliet@enpc.fr/Documents/2601_archetypes_re2020/data/processed/MC_runs/arch_mi_based_1000_gwp100.pkl", 'rb') as file:
    dict_mi = pickle.load(file)

df_bb = pl.read_parquet("/home/thibault.chevilliet@enpc.fr/Documents/2601_archetypes_re2020/data/processed/MC_runs/2026-5-9-9h53_MC_runs/scores.parquet")
df_var = pl.read_parquet("/home/thibault.chevilliet@enpc.fr/Documents/2601_archetypes_re2020/data/processed/Hb/20-06/scores_climate change_global warming potential (GWP100)_1000.parquet")


In [3]:
import bw2data as bd

bd.projects.set_current("2026-01-13-ecoinvent-3.11-cutoff")
arch_db = bd.Database("2026-4-22-16h25_archetypes_RE2020_BB")
dict_w = {}

for a in tqdm(arch_db) :
    phi = list(a.production())[0]["amount"]
    tmp = {e["input"][1] : e["amount"]/phi for e in tqdm(a.technosphere(),leave = False)}
    dict_w = dict_w | tmp

18:25:19+0200 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


100%|██████████| 5/5 [00:01<00:00,  3.20it/s]


In [4]:
# ajout d'une colonne poids
df_var = df_var.with_columns(
    poids = pl.col("batiment").replace_strict(dict_w, default = 0.,return_dtype = pl.Float64)
)

In [ ]:
# on a besoin d'afficher des tats pondérées, donc describe() est shunté pour x_var avec numpy 

In [ ]:
import math
def weighted_mean_std(values, weights):
    average = np.average(values, weights=weights)
    variance = np.average((values-average)**2, weights=weights)
    return (average, math.sqrt(variance))

In [5]:
usages_col = ["Maison individuelle ou accolée","Bureau","Logement collectif"]

In [18]:
from cmcrameri import cm
cmap = cm.berlin

palette = list(cmap(np.linspace(0.1, 0.9, 3)))

plt.rcParams.update({
    "text.usetex": True,
    "text.latex.preamble": r"\usepackage{amsmath}\usepackage{amssymb}",
    "font.family": "serif",
    "font.serif": "Computer Modern",
})


In [22]:
for i,usage_select in enumerate(usages_col) :
    df_mi_filtered = outliers(pl.DataFrame(list(dict_mi[usage_select][0])[:1000]),"column_0").filter(pl.col("extreme_outliers")==False)
    df_var_filtered = outliers(df_var.filter(pl.col("usage")==usage_select),"score").filter(pl.col("extreme_outliers")==False).select("score","poids")
    df_bb_filtered = outliers(df_bb.filter(pl.col("demand")== "_".join(usage_select.lower().split(" "))),"score").filter(pl.col("extreme_outliers")==False)

    fig,ax = plt.subplots()
    sns.histplot(df_mi_filtered,x= "column_0",ax = ax,label = "Archétype par intensités",stat="probability",
    element = "step",
    edgecolor='face',
    common_bins=False,color="black",common_norm = False, binwidth=20,alpha = 0.6,kde = False)
    sns.histplot(data = df_var_filtered,x = "score", weights="poids",ax = ax,label = "Archétype avec variabilité",stat="probability",
    element = "step",
    # edgecolor='face',
    common_bins=False,color=palette[i],common_norm = False, binwidth=20,alpha = 0.7,kde = False)
    sns.histplot(df_bb_filtered,x="score",ax = ax,label = "Archétype sans variabilité",stat="probability",
    element = "step",
    edgecolor="gray",
    common_bins=False,color=palette[i],common_norm = False, binwidth=20,alpha = 0.2,kde = False)
    ax.set_title(usage_select)
    ax.set_xlabel(r'$\text{kgCO}_2\text{ eq}$')
    ax.set_ylabel("Probabilité")
    ax.legend()
    plt.savefig(f"/home/thibault.chevilliet@enpc.fr/Documents/2601_archetypes_re2020/reports/figures/Distributions/comparaisons archétypes/stats_comp_{usage_select}.pdf",dpi=300)
    plt.clf()
    desc_var = df_var_filtered.select("score").describe().rename({"score": "x_var"})
    desc_bb  = df_bb_filtered.select("score").describe().rename({"score": "x_bb"}).drop("statistic")
    desc_mi  = df_mi_filtered.select("column_0").describe().rename({"column_0": "x_mi"}).drop("statistic")

    stats = pl.concat([desc_var, desc_bb, desc_mi], how="horizontal")
    w_mean, w_std = weighted_mean_std(df_var_filtered["score"].to_list(),df_var_filtered["poids"].to_list())
    w_25 , w_50, w_75 = np.quantile(df_var_filtered["score"].to_list(),q = [0.25,0.5,0.75],weights = df_var_filtered["poids"].to_list(),method="inverted_cdf")
    stats[2,1] = w_mean
    stats[3,1] = w_std
    stats[5,1] = w_25
    stats[6,1] = w_50
    stats[7,1] = w_75
    stats.write_csv(f"/home/thibault.chevilliet@enpc.fr/Documents/2601_archetypes_re2020/reports/tables/comparaisons_archetypes/stats_comp_{usage_select}")


<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

In [16]:
stats

statistic,x_var,x_bb,x_mi
str,f64,f64,f64
"""count""",1.061673e6,1000.0,988.0
"""null_count""",0.0,0.0,0.0
"""mean""",1280.339896,1347.898547,1706.026686
"""std""",322.869451,57.474368,682.781829
"""min""",274.891815,1165.744077,572.351955
"""25%""",1071.77356,1309.368198,1237.77041
"""50%""",1282.808472,1343.996428,1541.548052
"""75%""",1471.113892,1383.131138,1996.7392
"""max""",2817.708984,1598.658495,4363.980007


In [ ]:
# On remplace la colonne x_var avec les valeurs pondérées


In [19]:
pl.DataFrame(x_bb).describe()

statistic,column_0
str,f64
"""count""",1000.0
"""null_count""",0.0
"""mean""",1347.898547
"""std""",57.474368
"""min""",1165.744077
"""25%""",1309.368198
"""50%""",1343.996428
"""75%""",1383.131138
"""max""",1598.658495


100%|██████████| 5/5 [00:01<00:00,  3.52it/s]


In [41]:
len(dict_w)

79427

In [45]:
dict_w

{'566189_1': 0.0014442047165336203,
 '574565_1': 0.0010055524280477193,
 '562869_1': 0.0034005424180196643,
 '391295_1': 0.002180385152160839,
 '625510_1': 0.0007791014031894399,
 '459945_1': 0.0009928478216493016,
 '601256_1': 0.0004515285787545674,
 '399777_1': 0.0002130596829788662,
 '600205_1': 0.00043161324980569676,
 '579081_1': 0.0003301480824886058,
 '588126_1': 0.002610968298745042,
 '415508_1': 0.0009605712540425113,
 '558264_1': 0.0023503521837072348,
 '567604_1': 0.0003241391470298948,
 '562347_1': 0.0006019236490925906,
 '478887_1': 0.0002614745343890518,
 '572612_1': 0.0006216672941712125,
 '581794_1': 0.0014215424456607674,
 '469069_1': 0.0015751995095335193,
 '487797_1': 0.0008841719317817574,
 '570107_1': 0.00016103947029345408,
 '437840_1': 0.0002867120633156379,
 '505929_1': 0.0027795618593294473,
 '549036_1': 8.67003544756869e-05,
 '594516_1': 0.0008601361899469136,
 '605790_2': 0.0002135747345896129,
 '588186_1': 0.00017391576056212044,
 '606913_1': 0.0005500751202

In [51]:
missing = df_var.filter(
    ~pl.col("batiment").is_in(list(dict_w.keys()))
)["batiment"].unique()
print(missing)

shape: (217,)
Series: 'batiment' [cat]
[
	"352869_1"
	"495998_1"
	"478777_1"
	"505810_1"
	"351163_1"
	…
	"431160_1"
	"367644_1"
	"351647_1"
	"488647_1"
	"514648_1"
]
